# EMI Conduction velocity II

In this notebook, we will adapt the chain-of-cells example from the previous notebook by introducing a non-uniform distribution of sodium channels $g_{\rm Na}$.  

**Note:** The initial setup in this notebook is identical to the uniform case. Feel free to run through the first cells quickly until you reach the main exercise.

## FEniCS implementation

We begin by importing the necessary libraries. The Grandi ODE model is contained in `grandi.py`, and `emimesh.py` provides the Gmsh geometry definitions to build our 3D cell strand.


In [ ]:
import numpy as np

from dolfinx import fem, mesh, io, plot, default_scalar_type
from ufl import (
    inner,
    grad,
    TestFunctions,
    TrialFunctions,
    SpatialCoordinate,
    conditional,
    MixedFunctionSpace,
    extract_blocks,
    Measure,
)
import scifem
import pyvista

pyvista.set_jupyter_backend("static")

import grandi
from emimesh import generate_emi_mesh, scale, L_WE, step_x, step_y, step_z

### Mesh generation and submesh extraction

To keep computational times manageable, we generate a short strand of 6 cells. Using `scifem`, we extract dedicated submeshes for the shared extracellular space and each individual intracellular domain.

The extracellular space has cell tag 1, and intracellular domains are numbered $(2, 3, \ldots,  N_{cells} + 1)$.
We also generate specific facet markers: cell membranes $\Gamma_k$ are given unique tags $(10, 11, 12, \ldots)$, and gap junctions $\Gamma_{j,k}$ are marked separately $(100, 101, 102, \ldots)$.

In [ ]:
N_cells = 6

omega, ct, ft = generate_emi_mesh(N_cells=(6, 1, 1))


# Extract extracellular submesh
omega_e, e_to_p, _, _, _ = scifem.extract_submesh(omega, ct, 1)

# Extract intracellular submesh for each cell
omega_i, i_to_p = {}, {}
for k in range(N_cells):
    omega_i[k], i_to_p[k], _, _, _ = scifem.extract_submesh(omega, ct, 2 + k)

# Collect entity maps
entity_maps = [e_to_p] + [i_to_p[k] for k in range(N_cells)]

dx = Measure("dx", domain=omega, subdomain_data=ct)
dxE = dx(1)
dxI = {k: dx(2 + k) for k in range(N_cells)}

subdomain_data_list = []
for k in range(N_cells):
    subdomain_data_list.append(
        (10 + k, scifem.compute_interface_data(ct, ft.find(10 + k)).flatten())
    )
    subdomain_data_list.append(
        (20 + k, scifem.compute_interface_data(ct, ft.find(20 + k)).flatten())
    )
    if k < N_cells - 1:
        subdomain_data_list.append(
            (100 + k, scifem.compute_interface_data(ct, ft.find(100 + k)).flatten())
        )


gamma_mem, gamma_gap = {}, {}
for k in range(N_cells):
    gamma_mem[k] = scifem.find_interface(ct, 1, 2 + k)
    if k < N_cells - 1:
        gamma_gap[k] = scifem.find_interface(ct, 2 + k, 3 + k)

omega.topology.create_connectivity(omega.topology.dim - 1, omega.topology.dim)
num_facets = (
    omega.topology.index_map(omega.topology.dim - 1).size_local
    + omega.topology.index_map(omega.topology.dim - 1).num_ghosts
)
marker = np.full(num_facets, -1, dtype=np.int32)

for k in range(N_cells):
    marker[gamma_mem[k]] = 10 + k
    if k < N_cells - 1:
        marker[gamma_gap[k]] = 100 + k

marker_filter = np.flatnonzero(marker != -1).astype(np.int32)
ft_unique = mesh.meshtags(
    omega, omega.topology.dim - 1, marker_filter, marker[marker_filter]
)
ft_unique.name = "facet_tags"

### Model parameters
Next we define the model parameters. The values here follow those used in [Tveito et al. (2017)](https://doi.org/10.3389/fphy.2017.00048){cite}`emiconduction-tveito2017cell`.

In [ ]:
# Simulation parameters
dt_val = 0.002
sim_time = 2.0

# Physical parameters
sigma_e = fem.Constant(omega, 20.0)  # mS/cm
sigma_i = fem.Constant(omega, 5.0)  # mS/cm
dt = fem.Constant(omega, dt_val)  # ms
Cm = fem.Constant(omega, 1.0)  # µF/cm^2
C_gap = fem.Constant(omega, 1.0)  # µF/cm^2
R_gap = fem.Constant(omega, 0.0015)  # kΩcm2
g_Na_const = fem.Constant(omega, 23.0)  # mS/µF
v_rest = -85  # mV

### Function spaces and state variables

We assign a piecewise linear (CG1) function space to each individual submesh. To solve the coupled system simultaneously, we combine the spaces into a single `MixedFunctionSpace`. We also initialize the FEniCSx variables that will hold our current and previous spatial solutions, alongside the internal state variables for the Grandi ODE model.

In [ ]:
Ve = fem.functionspace(omega_e, ("Lagrange", 1))
Vi = {k: fem.functionspace(omega_i[k], ("Lagrange", 1)) for k in range(N_cells)}

# Build the mixed space
space_list = [Ve] + [Vi[k] for k in range(N_cells)]
W = MixedFunctionSpace(*space_list)

trial_functions = TrialFunctions(W)
test_functions = TestFunctions(W)

ue, ve = trial_functions[0], test_functions[0]
ui = {k: trial_functions[k + 1] for k in range(N_cells)}
vi = {k: test_functions[k + 1] for k in range(N_cells)}

# Initialize PDE states
ue_sol = fem.Function(Ve)
ui_sol = {k: fem.Function(Vi[k]) for k in range(N_cells)}
ue_n = fem.Function(Ve)
ui_n = {k: fem.Function(Vi[k]) for k in range(N_cells)}
t_act = {k: fem.Function(Vi[k]) for k in range(N_cells)}
for k in range(N_cells):
    ui_sol[k].x.array[:] = v_rest
    ui_n[k].x.array[:] = v_rest
    t_act[k].x.array[:] = -1.0

# Initialize Grandi states
grandi_states = {k: grandi.init_states(len(ui_sol[k].x.array)) for k in range(N_cells)}
d_states_dict = {k: np.zeros_like(grandi_states[k]) for k in range(N_cells)}
I_ion_func = {k: fem.Function(Vi[k]) for k in range(N_cells)}


### Integration measures

Next, we create the integration measures. We use `dx` for volume integration and `dS` for the internal interfaces (membranes and gap junctions).
The function `scifem.compute_interface_data` is used to define consistent positive (+) and negative (-) sides for each interface, which is required to ensure correct flux calculations.

In [ ]:
dx = Measure("dx", domain=omega, subdomain_data=ct)
dxE = dx(1)
dxI = {k: dx(2 + k) for k in range(N_cells)}

subdomain_data_list = []
for k in range(N_cells):
    subdomain_data_list.append(
        (10 + k, scifem.compute_interface_data(ct, ft_unique.find(10 + k)).flatten())
    )
    if k < N_cells - 1:
        subdomain_data_list.append(
            (
                100 + k,
                scifem.compute_interface_data(ct, ft_unique.find(100 + k)).flatten(),
            )
        )

dS = Measure("dS", domain=omega, subdomain_data=subdomain_data_list)

### Distribution of sodium channels

In homogenized models, the sodium conductance ($g_{Na}$) is typically treated as a uniform constant across the entire membrane. 
However, the EMI framework allows us to explicitly define non-uniform spatial distributions, such as clustering specific ion channels strictly at the ends of the cells. If we move the channels to the cell tips, the local density in those regions must increase to ensure the total number of channels, and therefore the overall conductance capacity of the cell, remains constant.

#### Exercise: Implement the non-uniform $g_{Na}$ distribution


```{exercise} Adjust the gNa distribution
:label: l15-gna-nonuniform

Complete the `TODO` sections in the code block below to correctly implement the spatially varying $g_{Na}$. 
   1. Compute the exact `peak_gNa` needed to conserve the total channel count. Remember to account for the channels consumed by the small `g_Na_baseline` in the middle of the cell.
   2. Scale the arrays to apply your new peak and baseline values. (Hint: The interpolated masks in `g_Na_vary_dict` currently hold `1.0` at the tips and `0.0` in the middle. You must mathematically map these values to `peak_gNa` and `g_Na_baseline`, respectively).
   3. Use the Pyvista code in the next cell to check your $g_{\rm Na}$ distribution. Verify that the channel density is highly concentrated at the cell tips, while the middle of the cell uniformly sits at the much lower baseline value.
```

In [ ]:
x = SpatialCoordinate(omega)


g_Na_baseline = 4.0  # Small non-zero baseline outside the tips
g_Na_dict = {}

area_clusters = 0.0
area_tot = 0.0

for k in range(N_cells):
    x_coords = omega_i[k].geometry.x[:, 0]
    x_min, x_max = np.min(x_coords), np.max(x_coords)

    # UFL conditional mask (1.0 at tips, 0.0 in middle)
    is_tip = conditional(
        x[0] <= x_min + L_WE + 1e-12,
        1.0,
        conditional(x[0] >= x_max - L_WE - 1e-12, 1.0, 0.0),
    )

    # Interpolate mask into the Function
    g_k = fem.Function(Vi[k])
    g_k.interpolate(fem.Expression(is_tip, Vi[k].element.interpolation_points))
    g_Na_dict[k] = g_k

    # Accumulate areas strictly on the membrane dS(10+k)
    area_clusters += fem.assemble_scalar(
        fem.form(g_k("-") * dS(10 + k), entity_maps=[i_to_p[k]])
    )
    # Total surface area of the cell membrane
    area_tot += fem.assemble_scalar(fem.form(1.0 * dS(10 + k)))

# ==============================================================================
# EXERCISE:
# 1. Calculate the area of the cell body (`area_body`).
# 2. Determine the total channel budget (`channels_total`).
# 3. Determine how much of the budget is consumed by the baseline (`channels_baseline`).
# 4. Calculate `peak_gNa` to conserve the remaining budget at the tips.
# 5. Scale `g_k.x.array[:]` to map the [0, 1] mask to [baseline, peak].
# ==============================================================================

# TODO: Conserve budget accounting for the non-zero baseline
# area_body = ...
# channels_total = ...
# channels_baseline = ...
# peak_gNa = ...

# print(f"Computed global cluster peak gNa: {peak_gNa:.2f} mS/µF")

# TODO: Scale all functions: map the [0, 1] mask to [baseline, peak]
# for g_k in g_Na_dict.values():
#     g_k.x.array[:] = ...

# ==============================================================================
# TEST: Verify channel conservation
# ==============================================================================
total_conductance_vary = 0.0
total_conductance_const = 0.0

for k in range(N_cells):
    total_conductance_vary += fem.assemble_scalar(
        fem.form(g_Na_dict[k]("-") * dS(10 + k), entity_maps=[i_to_p[k]])
    )
    total_conductance_const += fem.assemble_scalar(
        fem.form(g_Na_const * dS(10 + k), entity_maps=[i_to_p[k]])
    )

print("-" * 60)
print(f"Total conductance, constant gNa: {total_conductance_const:.4f}")
print(f"Total conductance, varying gNa:  {total_conductance_vary:.4f}")

if np.isclose(total_conductance_const, total_conductance_vary):
    print("SUCCESS: Total ion channel conductance is conserved.")
else:
    print("ERROR: Total channel conductance does not match.")
print("-" * 60)

In [ ]:
plotter = pyvista.Plotter(window_size=[1000, 200])
plotter.add_text("g_Na", font_size=14)

# Loop over each submesh and append its respective g_Na function to the plot
for k in range(N_cells):
    topology, cell_types, geometry = plot.vtk_mesh(Vi[k])
    pv_grid_k = pyvista.UnstructuredGrid(topology, cell_types, geometry)
    pv_grid_k.point_data["g_Na"] = g_Na_dict[k].x.array
    plotter.add_mesh(
        pv_grid_k,
        scalars="g_Na",
        cmap="plasma",
        show_edges=False,
        interpolate_before_map=False,
    )

plotter.camera.tight(padding=0.05, view="xy", adjust_render_window=False)
plotter.show()

### Stimulus current
To trigger the action potential, we apply an artificial stimulus current strictly to the left tip of the first cell. 

In [ ]:
I_stim_amp = fem.Constant(omega, 0.0)  # to be updated in loop
stim_mask = conditional(x[0] <= 24.0 * scale, 1.0, 0.0)

### Variational formulation

We can now assemble the standard weak form of the EMI equations.

In [ ]:
# Bulk terms
a = sigma_e * inner(grad(ue), grad(ve)) * dxE
for k in range(N_cells):
    a += sigma_i * inner(grad(ui[k]), grad(vi[k])) * dxI[k]
L = fem.Constant(omega, 0.0) * ve * dxE


# Membrane terms
for k in range(N_cells):
    v_m = ui[k]("-") - ue("+")
    w_v = vi[k]("-") - ve("+")
    v_m_n = ui_n[k]("-") - ue_n("+")

    stimulus = (I_stim_amp * stim_mask)("-") if k == 0 else fem.Constant(omega, 0.0)

    a += (Cm / dt) * v_m * w_v * dS(10 + k)
    L += ((Cm / dt) * v_m_n - I_ion_func[k]("-") + stimulus) * w_v * dS(10 + k)


# Gap junction terms
for k in range(N_cells - 1):
    w_jump = ui[k]("+") - ui[k + 1]("-")
    w_test = vi[k]("+") - vi[k + 1]("-")
    w_prev = ui_n[k]("+") - ui_n[k + 1]("-")

    a += (C_gap / dt + 1.0 / R_gap) * w_jump * w_test * dS(100 + k)
    L += (C_gap / dt) * w_prev * w_test * dS(100 + k)

### Boundary conditions

We close the system by grounding the extracellular potential at the outer boundary ($\partial\Omega$) to zero.

In [ ]:
coords_e = omega_e.geometry.x
xe_min, xe_max = np.min(coords_e[:, 0]), np.max(coords_e[:, 0])
ye_min, ye_max = np.min(coords_e[:, 1]), np.max(coords_e[:, 1])
ze_min, ze_max = np.min(coords_e[:, 2]), np.max(coords_e[:, 2])


def outer_boundary_locator(x):
    x_bounds = np.isclose(x[0], xe_min) | np.isclose(x[0], xe_max)
    y_bounds = np.isclose(x[1], ye_min) | np.isclose(x[1], ye_max)
    z_bounds = np.isclose(x[2], ze_min) | np.isclose(x[2], ze_max)
    return x_bounds | y_bounds | z_bounds


outer_facets = mesh.locate_entities_boundary(
    omega_e, omega_e.topology.dim - 1, outer_boundary_locator
)
bc_ground = fem.dirichletbc(
    default_scalar_type(0.0),
    fem.locate_dofs_topological(Ve, omega_e.topology.dim - 1, outer_facets),
    Ve,
)

### Linear problem and solve
Here, we configure the FEniCSx `LinearProblem` to assemble and solve the coupled block system for both the extra- and intracellular potentials.

In [ ]:
problem = fem.petsc.LinearProblem(
    extract_blocks(a),
    extract_blocks(L),
    u=[ue_sol] + [ui_sol[k] for k in range(N_cells)],
    bcs=[bc_ground],
    entity_maps=entity_maps,
    petsc_options={
        "ksp_type": "cg",
        "pc_type": "hypre",
        "ksp_rtol": 1e-6,
    },
    petsc_options_prefix="EMI_",
)

Before running the time loop, we define a Discontinuous Galerkin (DG) function to collect our outputs for writing to file, and we set up point-tracking in the middle of Cell 3 and Cell 5 to later calculate the macroscopic conduction velocity.

In [ ]:
# Combined solution for output
V_dg = fem.functionspace(omega, ("DG", 0))
u_out = fem.Function(V_dg, name="u")

vtx_file = io.VTXWriter(
    omega.comm, "output/EMI_chain_potential_nonuniform.bp", [u_out], engine="BP4"
)

local_cells_e = np.arange(
    omega_e.topology.index_map(omega_e.topology.dim).size_local, dtype=np.int32
)
local_cells_i = {
    k: np.arange(
        omega_i[k].topology.index_map(omega_i[k].topology.dim).size_local,
        dtype=np.int32,
    )
    for k in range(N_cells)
}

### Tracking wave propagation

To calculate the macroscopic conduction velocity (how fast the wave travels across the tissue), we need to record the exact time the action potential passes specific locations. 

In the snippet below, we set up tracking for the geometric centers of two adjacent cells further down the strand (Cell 3 and Cell 4). The variables `t1_val` and `t2_val` will store the activation times, initialized to `-1.0` to indicate the wave has not yet arrived.

In [ ]:
cv_cell_1, cv_cell_2 = 3, 4
eval_pt_1 = np.array([(cv_cell_1 + 0.5) * step_x, (step_y / 2.0), (step_z / 2.0)])
eval_pt_2 = np.array([(cv_cell_2 + 0.5) * step_x, (step_y / 2.0), (step_z / 2.0)])
t1_val, t2_val, t = -1.0, -1.0, 0.0

Now, we are ready to run the main simulation loop. We now pass the full array of $g_{\rm Na}$ values to the Grandi model. 

In [ ]:
print("Starting simulation...")

for i in range(int(sim_time / dt_val)):
    t += dt_val
    I_stim_amp.value = 800.0 if (t >= 0.0 and t <= 1.0) else 0.0

    # 1. Step the FEniCSx Spatial PDE
    problem.solve()
    ue_n.x.array[:] = ue_sol.x.array[:]

    for k in range(N_cells):
        V_m = ui_sol[k].x.array

        # 2. Step the Grandi model
        I_tot = grandi.step_grandi(
            grandi_states[k], V_m, g_Na_dict[k].x.array, dt_val, d_states_dict[k]
        )
        I_ion_func[k].x.array[:] = I_tot

        # 3. Tracking
        unactivated_mask = t_act[k].x.array < -0.5
        firing_mask = ui_sol[k].x.array >= 0.0
        t_act[k].x.array[unactivated_mask & firing_mask] = t
        ui_n[k].x.array[:] = ui_sol[k].x.array[:]

    v1 = scifem.evaluate_function(ui_sol[cv_cell_1], [eval_pt_1])[0, 0]
    v2 = scifem.evaluate_function(ui_sol[cv_cell_2], [eval_pt_2])[0, 0]
    if t1_val < 0.0 and v1 >= 0.0:
        t1_val = t
    if t2_val < 0.0 and v2 >= 0.0:
        t2_val = t
    if i % 25 == 0:
        print(
            f"Time: {t:.3f} ms | C{cv_cell_1}: {v1:.1f} mV | C{cv_cell_2}: {v2:.1f} mV"
        )
        # Update u_out and write to file
        for k in range(N_cells):
            u_out.interpolate(
                ui_sol[k],
                cells0=local_cells_i[k],
                cells1=i_to_p[k].sub_topology_to_topology(
                    local_cells_i[k], inverse=False
                ),
            )
        vtx_file.write(t)

vtx_file.close()
print("Simulation complete.")

## Postprocessing

During the simulation, we recorded the activation time (time when potential exceeded 0 mV) for every node in the mesh in the `t_act` array. 
Run the PyVista code block below to visualize the full activation wave.

In [ ]:
plotter = pyvista.Plotter(window_size=[1200, 200])
plotter.add_text("Action potential propagation", font_size=12)

for k in range(N_cells):
    topology, cell_types, geometry = plot.vtk_mesh(Vi[k])
    grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

    # Check if the cell has fired. If not, plot gray cell.
    if np.all(t_act[k].x.array > 0):
        grid.point_data["Activation Time (ms)"] = t_act[k].x.array
        plotter.add_mesh(grid, scalars="Activation Time (ms)")
    else:
        plotter.add_mesh(grid, color="gray", show_edges=False)

plotter.camera.tight(padding=0.05, view="xy", adjust_render_window=False)
plotter.show()

### Postprocessing: Computing conduction velocity

To evaluate how fast the electrical signal travels across our cell strand, we calculate the macroscopic conduction velocity. This metric provides the average wave speed over a multi-cell distance. 

How does the value here compare to the one from the uniform case?

In [ ]:
if t1_val > 0 and t2_val > 0:
    cv = ((eval_pt_2[0] - eval_pt_1[0]) / (t2_val - t1_val)) * 1000.0
    print(
        f"Activation at C{cv_cell_1}: {t1_val:.3f} ms | C{cv_cell_2}: {t2_val:.3f} ms"
    )
    print(f"Calculated Macroscopic CV: {cv:.2f} cm/s.")
else:
    print("Warning: Wave did not reach both cells.")

Run the final plotting cell below to visualize this microscopic velocity profile. Do you notice a difference compared to the uniform case?

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
import numpy as np

cells_to_plot = range(0, N_cells - 2)

fig, ax1 = plt.subplots(figsize=(6, 4))

ax1.set_xlabel("x (µm)")
ax1.set_ylabel("Activation Time (ms)", color="tab:blue")
ax1.tick_params("y", colors="tab:blue")

ax2 = ax1.twinx()
ax2.set_ylabel("Conduction Velocity (cm/s)", color="tab:orange")
ax2.tick_params("y", colors="tab:orange")

for k in cells_to_plot:
    # Get coordinates and activation times
    x, t = Vi[k].tabulate_dof_coordinates()[:, 0], t_act[k].x.array
    mask = t >= 0.0
    idx = np.argsort(x[mask])
    x_cm, t_val = x[mask][idx], t[mask][idx]

    x_um = x_cm * 10000.0  # convert to µm

    # Cell labels & shading
    ax1.text(
        x_um.mean(),
        0.02,
        f"Cell {k}",
        transform=ax1.get_xaxis_transform(),
        ha="center",
        color="gray",
    )
    if k % 2 == 0:
        ax1.axvspan(x_um[0], x_um[-1], color="gray", alpha=0.1, zorder=0)

    # Calculate CV
    spline = UnivariateSpline(x_cm, t_val, k=2, s=0.001)
    cv = 1000.0 / np.maximum(spline.derivative()(x_cm), 1e-8)

    ax1.plot(x_um, t_val, "ko", markersize=2, alpha=0.05)
    ax1.plot(x_um, spline(x_cm), color="tab:blue", linewidth=2)
    ax2.plot(x_um, cv, color="tab:orange", linewidth=2)

ax2.set_ylim(0, 1000)
plt.title("Activation & Conduction Velocity (Non-uniform gNa)")
plt.savefig("output/EMI_chain_CV_nonuniform.png")
plt.show()

## References
```{bibliography}
   :filter: cited
   :labelprefix:
   :keyprefix: emiconduction-
```